In [38]:
import pandas as pd
import re
import numpy as np
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
import string
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
from tensorflow.keras.layers import Bidirectional
from tensorflow.keras.layers import GlobalAveragePooling1D
from tensorflow.keras.layers import Dropout, Dense, Embedding, LSTM, Layer, Concatenate, Input
from sklearn.metrics import precision_recall_fscore_support
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

In [39]:
# Preprocess the sequences
def preprocess_sequence(sequence, type):
    if type == "nl_clip":
      # Convert to lowercase
      sequence = sequence.lower()
      # Remove single quotes
      cleaned_sequence = sequence.replace("'", " ")
      # Fix space issues (remove repeating spaces and leading/trailing spaces)
      sequence = re.sub(r'\s+', ' ', cleaned_sequence).strip()
      # Tokenize the sequence
      words = sequence.split()
      return words

    else:  # in case of regexp
      sequence = sequence.strip()
      return sequence

# Get all the unique words
def get_vocab(sequences):
    # Flatten all sequences into a single list of words
    all_words = [word for sequence in sequences for word in sequence]
    # Get unique words to build the vocabulary
    vocab = sorted(set(all_words))
    return vocab


In [40]:
# *** Reading data
csv_file = '/content/nl-rx-synth.csv'
data = pd.read_csv(csv_file)


# *** Data Preprocessing
# print('Preprocessing sequences...')
src_sequences = []
targ_sequences = []

for row in data.itertuples(index=False):
    src_sequence = preprocess_sequence(row[0], "nl_clip") if isinstance(row[0], str) else ""
    targ_sequence = preprocess_sequence(row[1], "regexp") if isinstance(row[0], str) else ""
    src_sequences.append(src_sequence)
    targ_sequences.append(targ_sequence)
print(f"Source sequences : {len(src_sequences)}\nTarget sequences : {len(targ_sequences)}")

# Create the vocabulary from source and target sequences
vocab = get_vocab(src_sequences + targ_sequences)

vocab_size = len(vocab)
print('Vocabulary size:', vocab_size)

# *** Creating Numerical representations
# Create a mapping of word to index and index to word
word2index = {word: index for index, word in enumerate(vocab)}
index2word = {index: word for index, word in enumerate(vocab)}
# Convert the source and target sequences to numerical representations
src_sequences_encoded = [[word2index[word] for word in sequence] for sequence in src_sequences]
targ_sequences_encoded = [[word2index[word] for word in sequence] for sequence in targ_sequences]
print(f"Source sequence encoded : {len(src_sequences_encoded)}\nTarget sequence encoded : {len(targ_sequences_encoded)}")


# *** Adding padding to make sequences of same length
print('Padding sequences...')
MAX_SEQ_LENGTH = 50  # Define the maximum sequence length for source and target sequences
src_sequences_padded = pad_sequences(src_sequences_encoded, maxlen=MAX_SEQ_LENGTH, padding='post')
targ_sequences_padded = pad_sequences(targ_sequences_encoded, maxlen=MAX_SEQ_LENGTH, padding='post')
# Convert the padded sequences to numpy arrays
src_sequences_padded = np.array(src_sequences_padded)
targ_sequences_padded = np.array(targ_sequences_padded)
print(f"Source sequences padded : {len(src_sequences_padded)}\nTarget sequences padded : {len(targ_sequences_padded)}")


# *** Data splitting
src_train, src_test, targ_train, targ_test = train_test_split(
    np.array(src_sequences_padded), np.array(targ_sequences_padded),
    test_size=0.1, random_state=42
)

print("\nTraining and testing data stats after 80:20 division")
print("Size of training data :", len(src_train))
print("Size of test data :", len(src_test))

print("Index2Word :", index2word)


Source sequences : 2000
Target sequences : 2000
Vocabulary size: 85
Source sequence encoded : 2000
Target sequence encoded : 2000
Padding sequences...
Source sequences padded : 2000
Target sequences padded : 2000

Training and testing data stats after 80:20 division
Size of training data : 1800
Size of test data : 200
Index2Word : {0: '&', 1: '(', 2: ')', 3: '*', 4: '+', 5: ',', 6: '-', 7: '.', 8: '0', 9: '2', 10: '3', 11: '4', 12: '5', 13: '6', 14: '7', 15: '9', 16: 'A', 17: 'E', 18: 'I', 19: 'O', 20: 'U', 21: 'Z', 22: '[', 23: '\\', 24: ']', 25: 'a', 26: 'and', 27: 'at', 28: 'b', 29: 'before', 30: 'by', 31: 'c', 32: 'capital', 33: 'character', 34: 'character,', 35: 'contain', 36: 'containing', 37: 'd', 38: 'dog', 39: 'don', 40: 'e', 41: 'either', 42: 'ending', 43: 'followed', 44: 'g', 45: 'have', 46: 'having', 47: 'i', 48: 'k', 49: 'l', 50: 'lake', 51: 'least', 52: 'letter', 53: 'letter,', 54: 'lines', 55: 'lower-case', 56: 'more', 57: 'n', 58: 'not', 59: 'number', 60: 'number,', 61:

In [41]:
# Define the attention layer
class Attention(Layer):
    def __init__(self, units):
        super(Attention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, query, values):
        # query shape == (batch_size, hidden size)
        # values shape == (batch_size, max_len, hidden size)

        # Expand dimensions to support addition
        query_with_time_axis = tf.expand_dims(query, 1)

        # Score shape == (batch_size, max_length, 1)
        score = self.V(tf.nn.tanh(
            self.W1(query_with_time_axis) + self.W2(values)))

        # Attention_weights shape == (batch_size, max_length, 1)
        attention_weights = tf.nn.softmax(score, axis=1)

        # context_vector shape after sum == (batch_size, hidden_size)
        context_vector = attention_weights * values
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

In [42]:
# Define the model architecture
embedding_dim = 128
lstm_units = 256

inputs = Input(shape=(MAX_SEQ_LENGTH,))
embedding_layer = Embedding(input_dim=vocab_size, output_dim=embedding_dim)(inputs)

lstm = LSTM(units=lstm_units, return_sequences=True)(embedding_layer)
dropout = Dropout(0.2)(lstm)
lstm2 = LSTM(units=lstm_units, return_sequences=True)(dropout)
dropout2 = Dropout(0.2)(lstm2)

# Apply attention to the second LSTM layer's output
context_vector, attention_weights = Attention(lstm_units)(query=dropout2, values=lstm)

output = Dense(units=vocab_size, activation='softmax')(context_vector)

model = keras.Model(inputs=inputs, outputs=output)

# Compile the model
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [43]:
# Train the model
num_epochs = 100
batch_size = 32

print('Training the model...')
history = model.fit(src_train, targ_train, epochs=num_epochs, batch_size=batch_size, validation_data=(src_test, targ_test))

# Plot the loss and accuracy graphs
plt.figure(figsize=(12, 6))

# Loss graph
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')

# Accuracy graph
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')

plt.tight_layout()
plt.show()

# *** Evaluate the model on the test set
print('Evaluating the model on the test set...')
loss, accuracy = model.evaluate(src_test, targ_test)
print('Test Loss:', loss)
print('Test Accuracy:', accuracy)

Training the model...
Epoch 1/100
51/57 [=========================>....] - ETA: 4s - loss: 2.5212 - accuracy: 0.4984

KeyboardInterrupt: 

In [ ]:
# Save the trained model
model_name = "nl_rx_synth"
model.save(f'/content/drive/MyDrive/Exp/Reg Exp/Models/regexp with attention/{model_name}.h5')
print(f'Model saved as {model_name}.h5')

Model saved as nl_rx_synth.h5


In [ ]:
model_name = "nl_rx_synth"
# Loading Model
model = keras.models.load_model(f'/content/drive/MyDrive/Exp/Reg Exp/Models/{model_name}.h5')

OSError: No file or directory found at /content/drive/MyDrive/Exp/Reg Exp/Models/nl_rx_synth.h5

In [46]:
def decode_data(sequences, data_type):
  if data_type == "test_preds":
      # Decode the predicted sequences
      decoded_test_predictions = []
      for prediction_sequence in sequences:
        decoded_sequence = [index2word[np.argmax(pred)] for pred in prediction_sequence]
        decoded_test_predictions.append(decoded_sequence)
      return decoded_test_predictions
  else:  # in case of ground truth
    # Convert the ground truth sequences to words
    decoded_ground_truth = []
    for ground_truth_sequence in sequences:
      decoded_sequence = [index2word[idx] for idx in ground_truth_sequence]
      decoded_ground_truth.append(decoded_sequence)
    return decoded_ground_truth

# Decode ground truth
def decode_input(sentences):
  decoded = []
  for sentence in sentences:
    decoded_sentence = []
    for index in sentence:
      decoded_sentence.append(index2word[index])
    decoded.append(decoded_sentence)
  return decoded


In [45]:
# Generate predictions for the entire test set
test_predictions = model.predict(src_test)
print(f"Predictions done on {len(src_test)} instances test data")

7/7 [==============================] - 3s 212ms/step
Predictions done on 200 instances test data


In [48]:
# Decode the predicted sequences
decoded_test_predictions = decode_data(test_predictions, "test_preds")
# Convert the ground truth sequences to words
decoded_ground_truth = decode_data(targ_test, "test_targ")
# Decode input nl query
decoded_src_test = decode_input(src_test)

print("NL Query :", decoded_src_test[100])
print("Prediction :", decoded_test_predictions[100])
print("Ground Truth :", decoded_ground_truth[100])

NL Query : ['lines', 'with', 'a', 'capital', 'letter', 'before', 'the', 'string', 'dog', 'or', 'the', 'string', 'truck', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&']
Prediction : ['&', '&', '&', ')', ')', ')', ')', ')', ')', ')', ')', ')', ')', ')', ')', ')', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&']
Ground Truth : ['(', '[', 'A', '-', 'Z', ']', ')', '.', '*', '(', '(', 'd', 'o', 'g', ')', '|', '(', 't', 'r', 'u', 'c', 'k', ')', ')', '.', '*', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&', '&']


In [ ]:
# print(src_test[0])
# print(decode_input([src_test[0]]))
# print(test_predictions[0])
# print(decoded_ground_truth[0])

In [ ]:
# Initialize lists to store per-sequence metrics
precisions = []
recalls = []
f1_scores = []

# Compute precision, recall, and F1-score for each sequence individually
for true_sequence, pred_sequence in zip(decoded_ground_truth, decoded_test_predictions):
    # Convert sequences to sets for easier comparison
    true_set = set(true_sequence)
    pred_set = set(pred_sequence)

    # Compute true positives, false positives, and false negatives
    true_positives = len(true_set.intersection(pred_set))
    false_positives = len(pred_set - true_set)
    false_negatives = len(true_set - pred_set)

    # Compute precision, recall, and F1-score for this sequence
    precision = true_positives / (true_positives + false_positives + 1e-9)  # Add a small value to avoid division by zero
    recall = true_positives / (true_positives + false_negatives + 1e-9)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-9)

    # Append to lists
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)

# Calculate average metrics
avg_precision = sum(precisions) / len(precisions)
avg_recall = sum(recalls) / len(recalls)
avg_f1 = sum(f1_scores) / len(f1_scores)

print('Average Precision:', avg_precision)
print('Average Recall:', avg_recall)
print('Average F1-Score:', avg_f1)

Average Precision: 0.9446585025118762
Average Recall: 0.8442534237566633
Average F1-Score: 0.8889380144477473


In [ ]:
# Manual input string
input_string = "lines containing the string 'dog' and the string 'truck'"

# Preprocess the input string
input_sequence = preprocess_sequence(input_string, "nl_clip")

# Convert the input sequence into numerical representation using the vocabulary
input_numerical = [word2index[word] for word in input_sequence if word in word2index]

# Pad the input sequence
input_padded = pad_sequences([input_numerical], maxlen=MAX_SEQ_LENGTH, padding='post')

# Get model prediction
predictions = model.predict(input_padded)

# Decode the predictions
predicted_sequence = [index2word[np.argmax(word_prob)] for word_prob in predictions[0] if np.argmax(word_prob) != 0]

# Convert the predicted sequence back into a string
predicted_string = ' '.join(predicted_sequence)

print("Input string:", input_string)
print("Predicted output:", predicted_string)

1/1 [==============================] - 0s 83ms/step
Input string: lines containing the string 'dog' and the string 'truck'
Predicted output: ( * d d o g g d r t t r u c k ) )


In [ ]:
# !pip install rouge
from rouge import Rouge
from nltk.translate.bleu_score import corpus_bleu

references = decoded_ground_truth
hypotheses = decoded_test_predictions

# Flatten the lists into strings
flat_references = [' '.join(sentences) for sentences in references]
flat_hypotheses = [' '.join(hypothesis) for hypothesis in hypotheses]
# flat_hypotheses = [' '.join(hypothesis.split()) for hypothesis in hypotheses]

# Create Rouge object
rouge = Rouge()

# Calculate ROUGE Scores
rouge_scores = rouge.get_scores(flat_hypotheses, flat_references, avg=True)

# Print ROUGE Scores
print(rouge_scores)

# Calculate BLEU Score for the entire corpus
bleu_score = corpus_bleu(references, hypotheses)
print(f"BLEU Score: {bleu_score:.4f}")

{'rouge-1': {'r': 0.8490280712751295, 'p': 0.9321927435599277, 'f': 0.8865636547274173}, 'rouge-2': {'r': 0.6802078571237322, 'p': 0.6855057854736702, 'f': 0.6817111838829077}, 'rouge-l': {'r': 0.8435098096454549, 'p': 0.9259195730170204, 'f': 0.8807106744893788}}
BLEU Score: 0.0000


/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_